## Services, DNS & service discovery

You ended notebook 03 with a Deployment running three nginx pods — each with its own IP, each IP minted fresh on every restart, rollout, and scale. Now a *web* pod wants to call your *api*. It can't hardcode pod IPs (they change), can't list-and-filter pods (that's a custom controller per client), and there's no per-pod DNS by default. What every client wants is a **stable name and address** pointing at "whichever api pods are alive right now."

That's the **Service** — the glue between consumers and producers:

```
 web pod  --http://api:80-->  Service (stable VIP)  -->  one of N ready api pods
              DNS: api → 10.96.0.42       kube-proxy rewrites to a live pod IP
```

The module builds Services from the ground up:

- **The abstraction** — a virtual IP + a **label selector**; the endpoints controller keeps the pod list fresh.
- **The three port numbers** — `port`, `targetPort`, `nodePort` — that the CKA wants you precise about.
- **Types** — `ClusterIP`, `NodePort`, `LoadBalancer`, `ExternalName` — how far the Service reaches.
- **How `kube-proxy` routes** — the ClusterIP is a *fiction* the kernel makes real.
- **Cluster DNS** — CoreDNS and the `<svc>.<ns>.svc.cluster.local` name pattern.
- **Headless Services** — DNS straight to pod IPs, the base under StatefulSets.

On our map this whole module lives in the **Networking** box — **Service**, **CoreDNS**, and the **kube-proxy** that wires them onto the Pods. Service is *the* foundational networking primitive; everything above it (Ingress, mesh) uses Services to find pods.